# SYNTHIA -> Cityscapes Final Resource Pipeline (2000 Images)

This notebook is the final low-resource run:

| Step | Description |
|------|-------------|
| 0 | Mount Drive, clone repo, install deps, set paths |
| 1 | Prepare SYNTHIA and Cityscapes data |
| 2 | Build multilabel JSONs for DAMP |
| 3 | Train/load the DAMP full checkpoint |
| 4 | Generate zero-shot, DAMP prompt-only, and DAMP full CAMs for 2000 SYNTHIA images |
| 5 | Build a class-wise hybrid CAM set from zero-shot + DAMP full |
| 6 | Evaluate zero-shot, prompt-only, DAMP full, and hybrid on the same 2000-image split |
| 7 | Generate pseudo masks from the selected CAM kind + selected post-processing method |
| 8 | Export image/mask pairs for segmentation training |

Default best source is `hybrid`. Cell 7 also prints a strict zero-shot baseline with fixed threshold for reporting, but that strict row is not used to auto-select pseudo masks.

In [ ]:
# ===== CELL 0a: MOUNT DRIVE =====
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ===== CELL 0b: CLONE / UPDATE REPO + INSTALL DEPS =====
import os, sys, subprocess

REPO_DIR = '/content/Damp_es'
REPO_URL = 'https://github.com/baominh5xx2/Damp_es_CS338.git'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'Repo already exists at {REPO_DIR}; updating with git pull --ff-only')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=False)
    pull = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=False)
    if pull.returncode != 0:
        print('WARNING: git pull failed, likely because the Colab repo has local edits.')
        print('If you need a clean update, run: !rm -rf /content/Damp_es and rerun this cell.')

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, check=False)

# Install dependencies.
!pip install -q timm yacs ftfy regex lxml ttach
!pip install -q opencv-python-headless scikit-learn matplotlib tqdm
!pip install -q datasets huggingface_hub pyarrow
!pip install -q git+https://github.com/KaiyangZhou/Dassl.pytorch.git



In [ ]:
# ===== CELL 0c: CONFIG - paths & settings =====
from pathlib import Path

# Change this to your Drive path if needed.
DATA_ROOT = Path('/content/drive/MyDrive/datasets/synthia_cs338')
OUTPUT_DIR = DATA_ROOT / 'output'

# Final low-resource run.
RUN_NAME = 'synthia_clipnorm_tau052_e3'
CAM_MAX_IMAGES = 2000
EVAL_MAX_IMAGES = CAM_MAX_IMAGES
GRID_SEARCH_MAX_IMAGES = 100  # choose threshold/alpha on this subset, then score exact on all EVAL_MAX_IMAGES
FAST_EVAL_DOWNSAMPLE = 4  # Cell 7 metric speedup: 4 means evaluate at 1/16 pixels. Set 1 for exact full-res.
EVAL_NUM_WORKERS = 16  # More parallel file loading/scoring. Raise to 24 if RAM/Drive is stable.
GRID_PARAM_WORKERS = 12  # Parallel threshold/alpha grid search workers.
BEST_CAM_KIND = 'hybrid'  # options: 'zero', 'prompt_only', 'damp_full', 'hybrid'
CRF_CONFIDENCE = 0.95
CRF_N_JOBS = 1
USE_CRF = False  # final-resource default: False is much faster
PSEUDO_MASK_THRESHOLD = 0.01  # fallback only; Cell 7 selects the final threshold/method

# Report-only strict zero-shot baseline: no grid-search/postprocess tuning.
ZERO_STRICT_ENABLED = True
ZERO_STRICT_METHOD = 'baseline'  # baseline, norm, boost, no_bg
ZERO_STRICT_THRES = 0.03
ZERO_STRICT_ALPHA = 1.0

# Derived paths.
SYNTHIA_RAW = DATA_ROOT / 'data' / 'raw' / 'synthia'
CITY_RAW = DATA_ROOT / 'data' / 'raw' / 'cityscapes'
PROCESSED = DATA_ROOT / 'data' / 'processed'

DAMP_DIR = OUTPUT_DIR / 'damp' / RUN_NAME
PROMPT_CKPT = DAMP_DIR / 'prompt_learner.pth'
CAM_ZERO_DIR = OUTPUT_DIR / 'synthia' / f'cams_zero_raw_{CAM_MAX_IMAGES}'
CAM_PROMPT_DIR = OUTPUT_DIR / 'synthia' / f'cams_damp_{RUN_NAME}_prompt_only_raw_{CAM_MAX_IMAGES}'
CAM_FULL_DIR = OUTPUT_DIR / 'synthia' / f'cams_damp_{RUN_NAME}_full_raw_{CAM_MAX_IMAGES}'
CAM_HYBRID_DIR = OUTPUT_DIR / 'synthia' / f'cams_hybrid_zero_full_{RUN_NAME}_{CAM_MAX_IMAGES}'
CAM_DIR_BY_KIND = {
    'zero': CAM_ZERO_DIR,
    'prompt_only': CAM_PROMPT_DIR,
    'damp_full': CAM_FULL_DIR,
    'hybrid': CAM_HYBRID_DIR,
}
BEST_CAM_DIR = CAM_DIR_BY_KIND[BEST_CAM_KIND]

SEG_EXPORT_DIR = OUTPUT_DIR / 'segmentation' / RUN_NAME
MASK_DIR = None  # set in Cell 8 after Cell 7 selects method/threshold
SEG_TRAIN_PAIRS = None  # set in Cell 8

SYNTHIA_IMG = SYNTHIA_RAW / 'images'
SYNTHIA_LBL = SYNTHIA_RAW / 'labels'
SYNTHIA_SPLIT = SYNTHIA_RAW / 'splits' / 'train.txt'
SYNTHIA_CAM_SPLIT = SYNTHIA_RAW / 'splits' / f'train_first{CAM_MAX_IMAGES}.txt'

CITY_IMG = CITY_RAW / 'images'
CITY_LBL = CITY_RAW / 'labels'
CITY_TRAIN_SPLIT = CITY_RAW / 'splits' / 'train.txt'
CITY_VAL_SPLIT = CITY_RAW / 'splits' / 'val.txt'

HF_SYNTHIA_REPO = 'Minhbao5xx2/synthia-rand-cityscapes-16class-parquet_fix'
HF_CITY_REPO = 'Chris1/cityscapes'

print(f'DATA_ROOT        : {DATA_ROOT}')
print(f'RUN_NAME         : {RUN_NAME}')
print(f'CAM_MAX_IMAGES   : {CAM_MAX_IMAGES}')
print(f'EVAL_MAX_IMAGES  : {EVAL_MAX_IMAGES}')
print(f'GRID_SEARCH_MAX  : {GRID_SEARCH_MAX_IMAGES}')
print(f'FAST_DOWNSAMPLE  : {FAST_EVAL_DOWNSAMPLE}')
print(f'EVAL_NUM_WORKERS : {EVAL_NUM_WORKERS}')
print(f'GRID_PARAM_WORKERS: {GRID_PARAM_WORKERS}')
print(f'DAMP_DIR         : {DAMP_DIR}')
print(f'PROMPT_CKPT      : {PROMPT_CKPT}')
print(f'CAM_ZERO_DIR     : {CAM_ZERO_DIR}')
print(f'CAM_PROMPT_DIR   : {CAM_PROMPT_DIR}')
print(f'CAM_FULL_DIR     : {CAM_FULL_DIR}')
print(f'CAM_HYBRID_DIR   : {CAM_HYBRID_DIR}')
print(f'BEST_CAM_KIND    : {BEST_CAM_KIND}')
print(f'BEST_CAM_DIR     : {BEST_CAM_DIR}')
print(f'ZERO_STRICT      : {ZERO_STRICT_ENABLED}, {ZERO_STRICT_METHOD}, t={ZERO_STRICT_THRES}, a={ZERO_STRICT_ALPHA}')
print(f'MASK_DIR         : {MASK_DIR}')
print(f'SEG_TRAIN_PAIRS  : {SEG_TRAIN_PAIRS}')

## Step 1: Download & Prepare SYNTHIA

Downloads from HuggingFace (fixed parquet with correct 16-bit labels), then converts to images/labels/splits.

In [ ]:
# ===== CELL 1: DOWNLOAD + PREPARE SYNTHIA =====
import os, glob
from pathlib import Path

PARQUET_DIR = DATA_ROOT / "synthia_parquet"

# ── 1a: Download parquet from HuggingFace ──
if not SYNTHIA_SPLIT.exists():
    print("Downloading SYNTHIA parquet from HuggingFace ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HF_SYNTHIA_REPO,
        repo_type="dataset",
        local_dir=str(PARQUET_DIR),
        max_workers=16,
    )
    print("Download complete.")
else:
    print("SYNTHIA data already prepared, skipping download.")

# ── 1b: Convert parquet → images/labels/splits ──
if not SYNTHIA_SPLIT.exists():
    print("Converting parquet to images/labels/splits ...")
    !python {REPO_DIR}/tools/prepare_synthia_hf.py \
        --parquet-dir {PARQUET_DIR / "parquet"} \
        --output-root {SYNTHIA_RAW} \
        --num-workers 16
else:
    print("SYNTHIA splits already exist, skipping conversion.")

# ── 1c: Verify ──
n_img = len(list(Path(SYNTHIA_IMG).glob("*.png"))) if SYNTHIA_IMG.exists() else 0
n_lbl = len(list(Path(SYNTHIA_LBL).glob("*.png"))) if SYNTHIA_LBL.exists() else 0
n_split = len(open(SYNTHIA_SPLIT).readlines()) if SYNTHIA_SPLIT.exists() else 0
print(f"\nSYNTHIA: {n_img} images, {n_lbl} labels, {n_split} split entries")

# Quick label sanity check
if n_lbl > 0:
    import cv2, numpy as np
    sample_lbl = sorted(Path(SYNTHIA_LBL).glob("*.png"))[0]
    arr = cv2.imread(str(sample_lbl), cv2.IMREAD_UNCHANGED)
    if arr.ndim == 3:
        unique = np.unique(arr[:,:,2])  # Red channel = class ID (16-bit)
    else:
        unique = np.unique(arr)
    n_classes = len([v for v in unique if v != 0])
    print(f"Label check ({sample_lbl.name}): {n_classes} valid classes, unique IDs: {unique.tolist()[:15]}")

## Step 2: Download & Prepare Cityscapes

In [ ]:
# ===== CELL 2: DOWNLOAD + PREPARE CITYSCAPES =====
if not CITY_VAL_SPLIT.exists():
    print("Downloading & preparing Cityscapes from HuggingFace ...")
    !python {REPO_DIR}/tools/prepare_cityscapes_hf.py \
        --dataset-id {HF_CITY_REPO} \
        --output-root {CITY_RAW} \
        --splits train,validation \
        --num-workers 16
else:
    print("Cityscapes already prepared.")

# Verify
n_city_img = len(list(Path(CITY_IMG).glob("*.png"))) if CITY_IMG.exists() else 0
n_city_train = len(open(CITY_TRAIN_SPLIT).readlines()) if CITY_TRAIN_SPLIT.exists() else 0
n_city_val = len(open(CITY_VAL_SPLIT).readlines()) if CITY_VAL_SPLIT.exists() else 0
print(f"Cityscapes: {n_city_img} images, {n_city_train} train, {n_city_val} val")

## Step 3: Build Multilabel JSON

Extracts per-image class labels from segmentation masks for multi-label classification.

In [ ]:
import os

# Patch dassl library to fix numpy compatibility error
dassl_file = '/usr/local/lib/python3.12/dist-packages/dassl/data/transforms/randaugment.py'
if os.path.exists(dassl_file):
    with open(dassl_file, 'r') as f:
        content = f.read()

    # Replace np.int with int
    new_content = content.replace('astype(np.int)', 'astype(int)')

    if content != new_content:
        with open(dassl_file, 'w') as f:
            f.write(new_content)
        print(f"Successfully patched {dassl_file}")
    else:
        print("File already patched or np.int not found.")
else:
    print(f"Could not find {dassl_file}. Please check installation path.")

In [ ]:
# ===== CELL 3: BUILD MULTILABEL JSON =====
SYNTHIA_ML_DIR = PROCESSED / "synthia_multilabel"
CITY_ML_DIR    = PROCESSED / "cityscapes_multilabel"

# ── SYNTHIA multilabel ──
synthia_ml_file = SYNTHIA_ML_DIR / "multilabel.json"
if not synthia_ml_file.exists():
    print("Building SYNTHIA multilabel ...")
    !python {REPO_DIR}/tools/build_synthia_multilabel.py \
        --split-file {SYNTHIA_SPLIT} \
        --label-dir {SYNTHIA_LBL} \
        --output-dir {SYNTHIA_ML_DIR} \
        --num-workers 16
else:
    print("SYNTHIA multilabel already exists.")

# ── Cityscapes train multilabel ──
city_train_ml = CITY_ML_DIR / "train_multilabel.json"
if not city_train_ml.exists():
    print("Building Cityscapes train multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_TRAIN_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file train_multilabel.json \
        --num-workers 16
else:
    print("Cityscapes train multilabel already exists.")

# ── Cityscapes val multilabel ──
city_val_ml = CITY_ML_DIR / "val_multilabel.json"
if not city_val_ml.exists():
    print("Building Cityscapes val multilabel ...")
    !python {REPO_DIR}/tools/build_cityscapes_multilabel.py \
        --split-file {CITY_VAL_SPLIT} \
        --label-dir {CITY_LBL} \
        --output-dir {CITY_ML_DIR} \
        --output-file val_multilabel.json \
        --num-workers 16
else:
    print("Cityscapes val multilabel already exists.")

print("\nAll multilabel files ready!")

## Step 4: Train or Load DAMP Full Checkpoint

`configs/trainers/damp_synthia_fast.yaml` is now the source of truth for the debugged run:

- 3 epochs for `prompt_learner` and `context_decoder`
- CLIP pixel normalization, not ImageNet normalization
- `TRAINER.DAMP.TAU = 0.52`
- `TRAINER.DAMP.PSEUDO_TEMP = 0.0`, meaning logits are calibrated by CLIP logit scale before sigmoid
- checkpoint every epoch

The generated `prompt_learner.pth` includes both `prompt_learner` and `context_decoder`, so CAM generation without `--damp_disable_decoder` is DAMP full.


In [ ]:
# ===== CELL 4: TRAIN / LOAD DAMP FULL =====
%cd {REPO_DIR}

if PROMPT_CKPT.exists():
    print(f'DAMP checkpoint already exists: {PROMPT_CKPT}')
    print('Delete the run directory if you want to retrain from scratch.')
else:
    print(f'Training DAMP full checkpoint -> {DAMP_DIR}')
    !python train.py \
        --config-file configs/trainers/damp_synthia_fast.yaml \
        DATASET.ROOT {DATA_ROOT} \
        OUTPUT_DIR {DAMP_DIR}

# Verify.
if PROMPT_CKPT.exists():
    import os
    size_mb = os.path.getsize(PROMPT_CKPT) / 1024 / 1024
    print(f'\nPrompt checkpoint: {PROMPT_CKPT} ({size_mb:.1f} MB)')
else:
    raise FileNotFoundError(f'prompt_learner.pth not found: {PROMPT_CKPT}')


## Step 5: Generate Zero-shot, DAMP Prompt-only, and DAMP Full CAMs for 2000 Images

This cell generates the CAM sets used in the final comparison:

- `zero`: CLIP-ES zero-shot CAMs.
- `prompt_only`: learned DAMP prompt only, with `context_decoder` disabled. This is an ablation, not the full method.
- `damp_full`: learned DAMP prompt plus `context_decoder`. This is the actual DAMP method.

CAM generation uses high resolution and attention refinement. Existing complete CAM folders are skipped. Delete a CAM folder if you need to regenerate it with the current settings.

In [ ]:
# ===== CELL 5: GENERATE ZERO-SHOT + DAMP PROMPT-ONLY + DAMP FULL CAMs (2000 images) =====
%cd {REPO_DIR}

import glob
import subprocess
from pathlib import Path

PROMPT_CKPT = DAMP_DIR / 'prompt_learner.pth'
if not PROMPT_CKPT.exists():
    raise FileNotFoundError(f'DAMP checkpoint not found: {PROMPT_CKPT}. Run Cell 4 first.')

# Create an explicit split so CAMs, metrics, masks, and train pairs match exactly.
SYNTHIA_CAM_SPLIT.parent.mkdir(parents=True, exist_ok=True)
with open(SYNTHIA_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]
entries_subset = entries[:CAM_MAX_IMAGES]
with open(SYNTHIA_CAM_SPLIT, 'w') as f:
    f.write('
'.join(entries_subset) + '
')
print(f'Wrote CAM split: {SYNTHIA_CAM_SPLIT} ({len(entries_subset)} entries)')

for d in (CAM_ZERO_DIR, CAM_PROMPT_DIR, CAM_FULL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Change these if you need a cheaper rerun.
MAX_LONG_SIDE = 2048
USE_REFINE = True
NUM_CAM_WORKERS = 4

def run_generate(args):
    if not USE_REFINE:
        args.append('--no_refine')
    print(' '.join(str(x) for x in args))
    subprocess.run([str(x) for x in args], check=True)

# 1. Zero-shot CLIP-ES baseline.
n_zero = len(glob.glob(str(CAM_ZERO_DIR / '*.npy')))
if n_zero >= CAM_MAX_IMAGES:
    print(f'Zero-shot CAMs already exist ({n_zero}).')
else:
    print(f'Generating zero-shot CAMs -> {CAM_ZERO_DIR} (max_long_side={MAX_LONG_SIDE})')
    run_generate([
        'python', 'generate_cams.py',
        '--dataset', 'synthia',
        '--img_root', SYNTHIA_IMG,
        '--label_root', SYNTHIA_LBL,
        '--split_file', SYNTHIA_CAM_SPLIT,
        '--cam_out_dir', CAM_ZERO_DIR,
        '--cam_score', 'softmax',
        '--max_images', CAM_MAX_IMAGES,
        '--max_long_side', MAX_LONG_SIDE,
        '--num_workers', NUM_CAM_WORKERS,
        '--skip_existing',
    ])

# 2. DAMP prompt-only ablation: disables context_decoder.
n_prompt = len(glob.glob(str(CAM_PROMPT_DIR / '*.npy')))
if n_prompt >= CAM_MAX_IMAGES:
    print(f'DAMP prompt-only CAMs already exist ({n_prompt}).')
else:
    print(f'Generating DAMP prompt-only CAMs -> {CAM_PROMPT_DIR} (max_long_side={MAX_LONG_SIDE})')
    run_generate([
        'python', 'generate_cams.py',
        '--dataset', 'synthia',
        '--img_root', SYNTHIA_IMG,
        '--label_root', SYNTHIA_LBL,
        '--split_file', SYNTHIA_CAM_SPLIT,
        '--cam_out_dir', CAM_PROMPT_DIR,
        '--damp_prompt_ckpt', PROMPT_CKPT,
        '--damp_name_mode', 'train',
        '--damp_disable_decoder',
        '--cam_score', 'raw',
        '--max_images', CAM_MAX_IMAGES,
        '--max_long_side', MAX_LONG_SIDE,
        '--num_workers', NUM_CAM_WORKERS,
        '--skip_existing',
    ])

# 3. DAMP full method: prompt learner + context_decoder.
n_full = len(glob.glob(str(CAM_FULL_DIR / '*.npy')))
if n_full >= CAM_MAX_IMAGES:
    print(f'DAMP full CAMs already exist ({n_full}).')
else:
    print(f'Generating DAMP full CAMs -> {CAM_FULL_DIR} (max_long_side={MAX_LONG_SIDE})')
    run_generate([
        'python', 'generate_cams.py',
        '--dataset', 'synthia',
        '--img_root', SYNTHIA_IMG,
        '--label_root', SYNTHIA_LBL,
        '--split_file', SYNTHIA_CAM_SPLIT,
        '--cam_out_dir', CAM_FULL_DIR,
        '--damp_prompt_ckpt', PROMPT_CKPT,
        '--damp_name_mode', 'train',
        '--cam_score', 'raw',
        '--max_images', CAM_MAX_IMAGES,
        '--max_long_side', MAX_LONG_SIDE,
        '--num_workers', NUM_CAM_WORKERS,
        '--skip_existing',
    ])

n_zero = len(glob.glob(str(CAM_ZERO_DIR / '*.npy')))
n_prompt = len(glob.glob(str(CAM_PROMPT_DIR / '*.npy')))
n_full = len(glob.glob(str(CAM_FULL_DIR / '*.npy')))
print(f'
zero CAMs       : {n_zero} in {CAM_ZERO_DIR}')
print(f'prompt-only CAMs: {n_prompt} in {CAM_PROMPT_DIR}')
print(f'DAMP full CAMs  : {n_full} in {CAM_FULL_DIR}')
if n_zero == 0 or n_prompt == 0 or n_full == 0:
    raise RuntimeError('Missing CAM files. Check generation logs above.')

## Step 6: Build Class-wise Hybrid CAMs

Hybrid is the last low-resource trick: use the source that was stronger by class.

- zero-shot: `road`, `building`, `wall`, `pole`, `car`, `bicycle`
- DAMP full: `sidewalk`, `fence`, `vegetation`, `person`, `rider`, `bus`
- other classes fall back to whichever source has the class.

This hybrid is `zero + DAMP full`; prompt-only remains only as an ablation row.

In [ ]:
# ===== CELL 6: MERGE ZERO-SHOT + DAMP FULL INTO HYBRID CAMs =====
import glob
import numpy as np
from pathlib import Path
from tqdm import tqdm

CAM_TYPE = 'attn_highres'
CAM_HYBRID_DIR.mkdir(parents=True, exist_ok=True)

# Cityscapes train IDs: 0 road, 1 sidewalk, 2 building, 3 wall, 4 fence,
# 5 pole, 8 vegetation, 11 person, 12 rider, 13 car, 15 bus, 18 bicycle.
DAMP_FULL_CLASSES = {1, 4, 8, 11, 12, 15}
ZERO_CLASSES = {0, 2, 3, 5, 13, 18}

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]

saved = 0
missing = []
for entry in tqdm(entries, desc='hybrid zero+damp_full'):
    stem = Path(entry).stem
    out_path = CAM_HYBRID_DIR / f'{stem}.npy'
    if out_path.exists():
        saved += 1
        continue

    zero_path = CAM_ZERO_DIR / f'{stem}.npy'
    damp_path = CAM_FULL_DIR / f'{stem}.npy'
    if not zero_path.exists() or not damp_path.exists():
        missing.append(stem)
        continue

    zero = np.load(zero_path, allow_pickle=True).item()
    damp = np.load(damp_path, allow_pickle=True).item()
    zero_keys = [int(x) for x in zero['keys'].tolist()]
    damp_keys = [int(x) for x in damp['keys'].tolist()]
    zero_map = {k: i for i, k in enumerate(zero_keys)}
    damp_map = {k: i for i, k in enumerate(damp_keys)}
    keys = sorted(set(zero_keys) | set(damp_keys))

    merged = []
    for k in keys:
        use_damp = k in DAMP_FULL_CLASSES
        if use_damp and k in damp_map:
            merged.append(damp[CAM_TYPE][damp_map[k]])
        elif (not use_damp) and k in zero_map:
            merged.append(zero[CAM_TYPE][zero_map[k]])
        elif k in damp_map:
            merged.append(damp[CAM_TYPE][damp_map[k]])
        elif k in zero_map:
            merged.append(zero[CAM_TYPE][zero_map[k]])

    if not merged:
        missing.append(stem)
        continue

    out = dict(damp)
    out[CAM_TYPE] = np.stack(merged, axis=0).astype(damp[CAM_TYPE].dtype, copy=False)
    out['keys'] = np.asarray(keys, dtype=np.int64)
    np.save(out_path, out)
    saved += 1

n_hybrid = len(glob.glob(str(CAM_HYBRID_DIR / '*.npy')))
print(f'Hybrid CAMs: {n_hybrid} in {CAM_HYBRID_DIR}')
if missing:
    print(f'WARNING: missing {len(missing)} hybrid entries. First 10: {missing[:10]}')
if n_hybrid == 0:
    raise RuntimeError('No hybrid CAMs generated.')

## Step 7: Fast Evaluate CAMs and Select Pseudo-mask Params

This cell searches post-processing params on a small calibration subset, then re-scores the selected params on all `EVAL_MAX_IMAGES`.

Speed notes:

- `FAST_EVAL_DOWNSAMPLE = 4` evaluates metrics on strided 1/16 pixels. This is much faster and good enough for choosing params.
- Set `FAST_EVAL_DOWNSAMPLE = 1` only if you need exact full-resolution CAM metrics for the report.
- `zero strict` is a fixed zero-shot baseline for reporting and is not used to auto-pick pseudo masks.

In [ ]:
# ===== CELL 7: FAST EVALUATE CAMs + STRICT ZERO REPORT =====
%cd {REPO_DIR}

import glob
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from cam.evaluate import entry_stem, resolve_label_path, map_mask_to_synthia16

N_CLASS = 19
CAM_TYPE = 'attn_highres'

# ---- Metrics ----

def fast_hist(label_true, label_pred, n_class):
    lt_all = label_true.ravel()
    lp_all = label_pred.ravel()
    mask = (lt_all >= 0) & (lt_all < n_class)
    lt = lt_all[mask].astype(np.int64, copy=False)
    lp = lp_all[mask].astype(np.int64, copy=False)
    lp[(lp < 0) | (lp >= n_class)] = n_class
    hist = np.bincount(
        (n_class + 1) * lt + lp,
        minlength=n_class * (n_class + 1),
    ).reshape(n_class, n_class + 1)
    return hist

def scores_from_hist(hist):
    tp = np.diag(hist[:, :N_CLASS])
    gt_count = hist.sum(axis=1)
    pred_count = hist[:, :N_CLASS].sum(axis=0)
    acc = tp.sum() / max(gt_count.sum(), 1.0)
    acc_cls = np.nanmean(tp / np.maximum(gt_count, 1.0))
    iu = tp / np.maximum(gt_count + pred_count - tp, 1.0)
    valid = gt_count > 0
    mean_iu = np.nanmean(iu[valid])
    freq = gt_count / max(gt_count.sum(), 1.0)
    fwavacc = (freq[freq > 0] * iu[freq > 0]).sum()
    return {
        'Pixel Accuracy': float(acc),
        'Mean Accuracy': float(acc_cls),
        'Mean IoU': float(mean_iu),
        'FW IoU': float(fwavacc),
        'Class IoU': dict(zip(range(N_CLASS), iu)),
    }

# ---- Fast CAM helpers ----

def maybe_downsample(cams, gt):
    ds = int(FAST_EVAL_DOWNSAMPLE)
    if ds > 1:
        cams = cams[:, ::ds, ::ds]
        gt = gt[::ds, ::ds]
    return cams, gt

def normalize_per_class(cams):
    out = cams.astype(np.float32, copy=False)
    flat = out.reshape(out.shape[0], -1)
    c_min = flat.min(axis=1)[:, None, None]
    c_max = flat.max(axis=1)[:, None, None]
    denom = c_max - c_min
    safe = denom > 1e-8
    norm = np.zeros_like(out, dtype=np.float32)
    np.divide(out - c_min, np.maximum(denom, 1e-8), out=norm, where=safe)
    return norm

def extrema(cams):
    idx = np.argmax(cams, axis=0).astype(np.int16, copy=False)
    maxv = np.take_along_axis(cams, idx[None, ...], axis=0)[0].astype(np.float32, copy=False)
    return maxv, idx

def load_eval_item(cam_dir, gt_root, entry):
    stem = entry_stem(entry)
    cam_path = Path(cam_dir) / f'{stem}.npy'
    gt_path = resolve_label_path(gt_root, entry)
    if not cam_path.exists() or not Path(gt_path).exists():
        return None
    d = np.load(str(cam_path), allow_pickle=True).item()
    if CAM_TYPE not in d:
        return None
    cams_raw = d[CAM_TYPE].astype(np.float32, copy=False)
    keys = d['keys'].astype(np.int64)
    gt = np.asarray(Image.open(gt_path), dtype=np.uint8)
    gt = map_mask_to_synthia16(gt)
    cams_raw, gt = maybe_downsample(cams_raw, gt)
    cams_norm = normalize_per_class(cams_raw)
    raw_max, raw_idx = extrema(cams_raw)
    norm_max, norm_idx = extrema(cams_norm)
    # Store max/argmax only; no full CAM arrays needed after this point.
    return {
        'keys': keys,
        'gt': gt,
        'raw_max': raw_max,
        'raw_idx': raw_idx,
        'norm_max': norm_max,
        'norm_idx': norm_idx,
    }

def pred_from_max_idx(maxv, idx, keys, thres=None, alpha=1.0, no_bg=False):
    pred = keys[idx].astype(np.uint8, copy=False)
    if no_bg:
        return pred
    if np.ndim(thres) == 0:
        fg = maxv > float(thres)
    else:
        fg = maxv > thres
    out = np.full(idx.shape, 255, dtype=np.uint8)
    out[fg] = pred[fg]
    return out

def predict_item(item, method, thres=0.0, alpha=1.0):
    keys = item['keys']
    if method == 'baseline':
        return pred_from_max_idx(item['raw_max'], item['raw_idx'], keys, thres)
    if method == 'norm':
        return pred_from_max_idx(item['norm_max'], item['norm_idx'], keys, thres)
    if method == 'boost':
        bg = thres * np.power(np.clip(1.0 - item['norm_max'], 0, 1), alpha).astype(np.float32, copy=False)
        return pred_from_max_idx(item['norm_max'], item['norm_idx'], keys, bg)
    if method == 'no_bg':
        return pred_from_max_idx(item['norm_max'], item['norm_idx'], keys, no_bg=True)
    raise ValueError(f'Unknown method: {method}')

def score_items(items, method, thres=0.0, alpha=1.0):
    hist = np.zeros((N_CLASS, N_CLASS + 1), dtype=np.float64)
    for item in items:
        pred = predict_item(item, method, thres, alpha)
        hist += fast_hist(item['gt'], pred, N_CLASS)
    return scores_from_hist(hist)

def parallel_grid_search(items, method, params, desc):
    def eval_one(param):
        thres, alpha = param
        miou = score_items(items, method, thres, alpha)['Mean IoU']
        return miou, float(thres), float(alpha)

    best_miou, best_thres, best_alpha = -1.0, 0.0, 1.0
    errors = []
    with ThreadPoolExecutor(max_workers=GRID_PARAM_WORKERS) as executor:
        futures = {executor.submit(eval_one, p): p for p in params}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc, leave=False):
            param = futures[fut]
            try:
                miou, thres, alpha = fut.result()
            except Exception as e:
                errors.append((param, repr(e)))
                if len(errors) <= 5:
                    print(f'[grid-error] {desc} param={param}: {type(e).__name__}: {e}')
                continue
            if miou > best_miou:
                best_miou, best_thres, best_alpha = miou, thres, alpha
    if errors:
        print(f'[grid-error] {desc}: {len(errors)} failed params. First errors: {errors[:3]}')
    if best_miou < 0:
        raise RuntimeError(f'All grid params failed for {desc}. First errors: {errors[:3]}')
    return best_miou, best_thres, best_alpha

def load_items_parallel(cam_dir, gt_root, entries, desc):
    items = []
    errors = []
    missing = 0
    with ThreadPoolExecutor(max_workers=EVAL_NUM_WORKERS) as executor:
        futures = {executor.submit(load_eval_item, cam_dir, gt_root, e): e for e in entries}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=desc):
            entry = futures[fut]
            try:
                item = fut.result()
            except Exception as e:
                errors.append((entry, repr(e)))
                if len(errors) <= 10:
                    print(f'[load-error] {desc} entry={entry}: {type(e).__name__}: {e}')
                continue
            if item is not None:
                items.append(item)
            else:
                missing += 1
    if missing:
        print(f'[load-warning] {desc}: skipped {missing} missing/invalid entries')
    if errors:
        print(f'[load-error] {desc}: {len(errors)} worker errors. First errors: {errors[:3]}')
    return items

def stream_score_protocols(cam_dir, split_file, gt_root, n_images, protocols):
    with open(split_file, 'r') as f:
        entries = [line.strip() for line in f if line.strip()][:n_images]

    def score_one(entry):
        item = load_eval_item(cam_dir, gt_root, entry)
        if item is None:
            return None
        local = {name: np.zeros((N_CLASS, N_CLASS + 1), dtype=np.float64) for name in protocols}
        for name, cfg in protocols.items():
            pred = predict_item(item, cfg['method'], cfg.get('thres', 0.0), cfg.get('alpha', 1.0))
            local[name] += fast_hist(item['gt'], pred, N_CLASS)
        return local

    hists = {name: np.zeros((N_CLASS, N_CLASS + 1), dtype=np.float64) for name in protocols}
    n_loaded = 0
    errors = []
    missing = 0
    with ThreadPoolExecutor(max_workers=EVAL_NUM_WORKERS) as executor:
        futures = {executor.submit(score_one, e): e for e in entries}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f'Full scoring {Path(cam_dir).name}'):
            entry = futures[fut]
            try:
                local = fut.result()
            except Exception as e:
                errors.append((entry, repr(e)))
                if len(errors) <= 10:
                    print(f'[score-error] {Path(cam_dir).name} entry={entry}: {type(e).__name__}: {e}')
                continue
            if local is None:
                missing += 1
                continue
            n_loaded += 1
            for name in protocols:
                hists[name] += local[name]

    if missing:
        print(f'[score-warning] {Path(cam_dir).name}: skipped {missing} missing/invalid entries')
    if errors:
        print(f'[score-error] {Path(cam_dir).name}: {len(errors)} worker errors. First errors: {errors[:3]}')
    if n_loaded == 0:
        raise RuntimeError(f'No valid entries scored for {cam_dir}. errors={errors[:3]} missing={missing}')

    return {name: scores_from_hist(hist) for name, hist in hists.items()}, n_loaded

# ---- Main evaluation ----

def boost_evaluate(cam_dir, split_file, gt_root, n_images, kind=None):
    with open(split_file, 'r') as f:
        entries_full = [line.strip() for line in f if line.strip()][:n_images]
    entries_grid = entries_full[:min(len(entries_full), GRID_SEARCH_MAX_IMAGES)]

    cache = load_items_parallel(
        cam_dir, gt_root, entries_grid, desc=f'Grid loading {Path(cam_dir).name}'
    )
    if not cache:
        print(f'  No data from {cam_dir}')
        return {}

    print(f'  Grid search images: {len(cache)}; final scoring images: {len(entries_full)}; downsample={FAST_EVAL_DOWNSAMPLE}x')

    flat_params = [(float(t), 1.0) for t in np.arange(0.005, 0.25, 0.005)]
    best_bl_miou, best_bl_thres, _ = parallel_grid_search(
        cache, 'baseline', flat_params, desc='Grid baseline'
    )

    best_norm_miou, best_norm_thres, _ = parallel_grid_search(
        cache, 'norm', flat_params, desc='Grid norm'
    )

    boost_params = [(float(t), float(alpha))
                    for alpha in [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]
                    for t in np.arange(0.05, 0.70, 0.05)]
    best_boost_miou, best_boost_thres, best_boost_alpha = parallel_grid_search(
        cache, 'boost', boost_params, desc='Grid boost'
    )

    nobg_grid_miou = score_items(cache, 'no_bg')['Mean IoU']

    protocols = {
        'baseline': {'method': 'baseline', 'thres': best_bl_thres, 'alpha': 1.0},
        'norm': {'method': 'norm', 'thres': best_norm_thres, 'alpha': 1.0},
        'boost': {'method': 'boost', 'thres': best_boost_thres, 'alpha': best_boost_alpha},
        'no_bg': {'method': 'no_bg', 'thres': 0.0, 'alpha': 0.0},
    }
    if kind == 'zero' and ZERO_STRICT_ENABLED:
        protocols['strict'] = {
            'method': ZERO_STRICT_METHOD,
            'thres': ZERO_STRICT_THRES,
            'alpha': ZERO_STRICT_ALPHA,
        }

    full_scores, n_loaded = stream_score_protocols(cam_dir, split_file, gt_root, n_images, protocols)

    print(f"  Baseline (raw, flat bg): mIoU={full_scores['baseline']['Mean IoU']:.4f} "
          f"(grid={best_bl_miou:.4f}, thres={best_bl_thres:.3f})")
    print(f"  Norm only (flat bg):     mIoU={full_scores['norm']['Mean IoU']:.4f} "
          f"(grid={best_norm_miou:.4f}, thres={best_norm_thres:.3f})")
    print(f"  Boost (norm+adaptive):   mIoU={full_scores['boost']['Mean IoU']:.4f} "
          f"(grid={best_boost_miou:.4f}, thres={best_boost_thres:.2f}, alpha={best_boost_alpha:.1f})")
    print(f"  No-bg (argmax fg):       mIoU={full_scores['no_bg']['Mean IoU']:.4f} "
          f"(grid={nobg_grid_miou:.4f})")
    if 'strict' in full_scores:
        print(f"  Zero strict report:      mIoU={full_scores['strict']['Mean IoU']:.4f} "
              f"(method={ZERO_STRICT_METHOD}, thres={ZERO_STRICT_THRES}, alpha={ZERO_STRICT_ALPHA})")

    candidates = [
        ('baseline', full_scores['baseline']['Mean IoU']),
        ('norm', full_scores['norm']['Mean IoU']),
        ('boost', full_scores['boost']['Mean IoU']),
        ('no_bg', full_scores['no_bg']['Mean IoU']),
    ]
    best_name, best_val = max(candidates, key=lambda x: x[1])
    print(f">>> BEST: {best_name} = {best_val:.4f} ({n_loaded} scored images)")

    result = {
        'baseline': {'miou': full_scores['baseline']['Mean IoU'], 'grid_miou': best_bl_miou, 'thres': best_bl_thres},
        'norm': {'miou': full_scores['norm']['Mean IoU'], 'grid_miou': best_norm_miou, 'thres': best_norm_thres},
        'boost': {'miou': full_scores['boost']['Mean IoU'], 'grid_miou': best_boost_miou, 'thres': best_boost_thres, 'alpha': best_boost_alpha},
        'no_bg': {'miou': full_scores['no_bg']['Mean IoU'], 'grid_miou': nobg_grid_miou},
        'strict': None,
        'best_method': best_name,
        'n_images': n_loaded,
    }
    if 'strict' in full_scores:
        result['strict'] = {
            'miou': full_scores['strict']['Mean IoU'],
            'method': ZERO_STRICT_METHOD,
            'thres': ZERO_STRICT_THRES,
            'alpha': ZERO_STRICT_ALPHA,
        }
    return result

# ---- Run for each CAM kind ----

all_results = {}
for kind, cam_dir in CAM_DIR_BY_KIND.items():
    n_cam = len(glob.glob(str(cam_dir / '*.npy')))
    print(' ' + '=' * 80)
    print(f'Evaluating {kind}: {cam_dir} ({n_cam} CAM files)')
    if n_cam == 0:
        print('  SKIP: no CAM files')
        continue
    all_results[kind] = boost_evaluate(
        str(cam_dir), str(SYNTHIA_CAM_SPLIT), str(SYNTHIA_LBL), EVAL_MAX_IMAGES, kind=kind
    )

# ---- Summary ----

print(' ' + '=' * 80)
print('SUMMARY: mIoU on EVAL_MAX_IMAGES after params selected on GRID_SEARCH_MAX_IMAGES')
print(f'Downsample factor for Cell 7 metrics: {FAST_EVAL_DOWNSAMPLE}x')
print('=' * 80)
print(f"  {'Kind':<14s} {'Baseline':>10s} {'Norm':>10s} {'Boost':>10s} {'No-bg':>10s} {'Strict':>10s}  Best")
print(f"  {'-'*14} {'-'*10} {'-'*10} {'-'*10} {'-'*10} {'-'*10}  {'-'*10}")
for kind in CAM_DIR_BY_KIND:
    r = all_results.get(kind, {})
    if not r:
        continue
    strict = r.get('strict')
    strict_str = f"{strict['miou']:.4f}" if strict else '-'
    print(f"  {kind:<14s} {r['baseline']['miou']:>10.4f} {r['norm']['miou']:>10.4f} "
          f"{r['boost']['miou']:>10.4f} {r['no_bg']['miou']:>10.4f} {strict_str:>10s}  {r['best_method']}")

# ---- Auto-pick best kind + method. Strict zero is report-only and excluded. ----

def result_best_miou(r):
    return max(r['baseline']['miou'], r['norm']['miou'], r['boost']['miou'], r['no_bg']['miou'])

best_kind = max(all_results.keys(), key=lambda k: result_best_miou(all_results[k]))
r = all_results[best_kind]
BEST_CAM_KIND = best_kind
BEST_CAM_DIR = CAM_DIR_BY_KIND[BEST_CAM_KIND]
BEST_METHOD = r['best_method']

if BEST_METHOD == 'boost':
    BEST_THRES = r['boost']['thres']
    BEST_ALPHA = r['boost']['alpha']
elif BEST_METHOD == 'norm':
    BEST_THRES = r['norm']['thres']
    BEST_ALPHA = 1.0
elif BEST_METHOD == 'baseline':
    BEST_THRES = r['baseline']['thres']
    BEST_ALPHA = 1.0
else:  # no_bg
    BEST_THRES = 0.0
    BEST_ALPHA = 0.0

print(f">>> BEST CAM KIND : {BEST_CAM_KIND}")
print(f"  >>> BEST METHOD   : {BEST_METHOD}")
print(f"  >>> BEST THRES    : {BEST_THRES}")
print(f"  >>> BEST ALPHA    : {BEST_ALPHA}")
print('Cell 8 will use these params to generate pseudo masks.')

## Step 8: Create Pseudo Masks from Best Params (Cell 7)

Use `BEST_CAM_KIND`, `BEST_METHOD`, `BEST_THRES`, and `BEST_ALPHA` from Cell 7 to generate pseudo masks.

The output folder includes method, threshold, and alpha in its name, so rerunning with new parameters will not accidentally reuse stale masks.

In [ ]:
# ===== CELL 8: SELECTED CAMs -> PSEUDO-MASKS (using best params from Cell 7) =====
%cd {REPO_DIR}

import glob
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from cam.evaluate import entry_stem

BEST_CAM_DIR = CAM_DIR_BY_KIND[BEST_CAM_KIND]
BEST_METHOD_SAFE = str(BEST_METHOD).replace('-', '_')
BEST_THRES_TAG = f"t{int(round(float(BEST_THRES) * 1000)):03d}"
BEST_ALPHA_TAG = f"a{str(BEST_ALPHA).replace('.', 'p')}"
MASK_DIR = OUTPUT_DIR / 'synthia' / f'pseudo_masks_{BEST_CAM_KIND}_{BEST_METHOD_SAFE}_{BEST_THRES_TAG}_{BEST_ALPHA_TAG}_{RUN_NAME}_{CAM_MAX_IMAGES}'
SEG_EXPORT_DIR = OUTPUT_DIR / 'segmentation' / RUN_NAME
SEG_TRAIN_PAIRS = SEG_EXPORT_DIR / f'train_pairs_{BEST_CAM_KIND}_{BEST_METHOD_SAFE}_{BEST_THRES_TAG}_{BEST_ALPHA_TAG}_first{CAM_MAX_IMAGES}.txt'

print(f'BEST_CAM_KIND  : {BEST_CAM_KIND}')
print(f'BEST_CAM_DIR   : {BEST_CAM_DIR}')
print(f'BEST_METHOD    : {BEST_METHOD}')
print(f'BEST_THRES     : {BEST_THRES}')
print(f'BEST_ALPHA     : {BEST_ALPHA}')
print(f'MASK_DIR       : {MASK_DIR}')
print(f'SEG_TRAIN_PAIRS: {SEG_TRAIN_PAIRS}')

MASK_DIR.mkdir(parents=True, exist_ok=True)
n_existing = len(glob.glob(str(MASK_DIR / '*.png')))
if n_existing >= CAM_MAX_IMAGES:
    print(f'Pseudo masks already exist ({n_existing}). Skipping.')
else:
    with open(SYNTHIA_CAM_SPLIT, 'r') as f:
        entries = [line.strip() for line in f if line.strip()][:CAM_MAX_IMAGES]

    saved = 0
    missing = []
    for entry in tqdm(entries, desc='pseudo masks'):
        stem = entry_stem(entry)
        cam_path = BEST_CAM_DIR / f'{stem}.npy'
        out_path = MASK_DIR / f'{stem}.png'
        if out_path.exists():
            saved += 1
            continue
        if not cam_path.exists():
            missing.append(stem)
            continue

        d = np.load(str(cam_path), allow_pickle=True).item()
        cams = d['attn_highres'].astype(np.float32)
        keys = d['keys'].astype(np.int64)

        if BEST_METHOD in ('boost', 'norm', 'no_bg'):
            for c in range(cams.shape[0]):
                c_min, c_max = cams[c].min(), cams[c].max()
                if c_max - c_min > 1e-8:
                    cams[c] = (cams[c] - c_min) / (c_max - c_min)
                else:
                    cams[c] = 0.0

        if BEST_METHOD == 'no_bg':
            idx = np.argmax(cams, axis=0)
            pred = keys[idx].astype(np.uint8)
        elif BEST_METHOD == 'boost':
            max_fg = np.max(cams, axis=0, keepdims=True)
            bg = BEST_THRES * np.power(np.clip(1.0 - max_fg, 0, 1), BEST_ALPHA).astype(cams.dtype)
            c = np.concatenate([bg, cams], axis=0)
            idx = np.argmax(c, axis=0)
            pred = np.full(idx.shape, 255, dtype=np.uint8)
            fg = idx > 0
            pred[fg] = keys[idx[fg] - 1].astype(np.uint8)
        else:  # baseline or norm
            bg = np.full((1, cams.shape[1], cams.shape[2]), BEST_THRES, dtype=cams.dtype)
            c = np.concatenate([bg, cams], axis=0)
            idx = np.argmax(c, axis=0)
            pred = np.full(idx.shape, 255, dtype=np.uint8)
            fg = idx > 0
            pred[fg] = keys[idx[fg] - 1].astype(np.uint8)

        Image.fromarray(pred.astype(np.uint8)).save(out_path)
        saved += 1

    if missing:
        print(f'WARNING: missing {len(missing)} CAM files. First 10: {missing[:10]}')
    print(f'Saved/kept {saved} pseudo masks.')

n_masks = len(glob.glob(str(MASK_DIR / '*.png')))
print(f'
{n_masks} pseudo masks saved to {MASK_DIR}')
if n_masks == 0:
    raise RuntimeError('No pseudo masks generated.')

## Step 9: Export Segmentation Training Pairs

This writes `image_path mask_path` pairs for the selected pseudo-mask set.


In [ ]:
# ===== CELL 9: EXPORT IMAGE/MASK PAIRS FOR SEGMENTATION TRAINING =====
from pathlib import Path

SEG_EXPORT_DIR = OUTPUT_DIR / 'segmentation' / RUN_NAME
SEG_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

with open(SYNTHIA_CAM_SPLIT, 'r') as f:
    entries = [line.strip() for line in f if line.strip()]

pairs = []
missing = []
for entry in entries:
    stem = Path(entry).stem
    image_path = SYNTHIA_IMG / f'{stem}.png'
    if not image_path.exists():
        image_path = SYNTHIA_IMG / Path(entry).name
    mask_path = MASK_DIR / f'{stem}.png'

    if image_path.exists() and mask_path.exists():
        pairs.append((image_path, mask_path))
    else:
        missing.append((image_path, mask_path))

with open(SEG_TRAIN_PAIRS, 'w') as f:
    for image_path, mask_path in pairs:
        f.write(f'{image_path} {mask_path}\n')

print(f'Exported {len(pairs)} image/mask pairs -> {SEG_TRAIN_PAIRS}')
if missing:
    print(f'WARNING: {len(missing)} entries missing image or mask. First 5:')
    for image_path, mask_path in missing[:5]:
        print(' ', image_path, '|', mask_path)

print('\nSegmentation training inputs:')
print(f'  image_dir : {SYNTHIA_IMG}')
print(f'  mask_dir  : {MASK_DIR}')
print(f'  pair_file : {SEG_TRAIN_PAIRS}')
print(f'  ignore_id : 255')
print(f'  classes   : Cityscapes train IDs, SYNTHIA-valid subset')
